In [43]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, sqrt, atan2

In [48]:
# Load route-stop table
route_stops = pd.read_csv("route_stops_full.csv")

# Standardize column names once
route_stops = route_stops.rename(columns={
    "Stop Name": "stop_name",
    "PublishedLineName": "route",
    "DirectionRef": "direction"
})

route_stops.head()

,route,direction,stop_name,stoplat,stoplon,obs
0,B35,0.0,14 AV/36 ST,40.641006,-73.982306,925
1,B35,0.0,14 AV/39 ST,40.639481,-73.984300,1408
2,B35,0.0,14 AV/CHURCH AV,40.641736,-73.981549,1037
3,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784
4,B35,0.0,39 ST/12 AV,40.642054,-73.988660,997


In [49]:
# Build node table
nodes = (
    route_stops.groupby("stop_name", as_index=False)
    .agg(
        stoplat=("stoplat", "median"),
        stoplon=("stoplon", "median"),
        totalobs=("obs", "sum")
    )
)

nodes.head()

,stop_name,stoplat,stoplon,totalobs
0,108 ST/53 AV,40.742945,-73.854753,4215
1,108 ST/HORACE HARDING EP,40.737824,-73.851798,2983
2,108 ST/HORACE HARDING EXPWY,40.738070,-73.852257,2833
3,108 ST/MARTENSE AV,40.742284,-73.854417,1034
4,108 ST/OTIS AV,40.741549,-73.854038,2345


#### Approach 2

In [50]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

In [51]:
# Build route-level edges first
route_edges = []

for (route, direction), group in route_stops.groupby(["route", "direction"]):
    group = group.drop_duplicates(subset=["stop_name"]).copy()

    lonspan = group["stoplon"].max() - group["stoplon"].min()
    latspan = group["stoplat"].max() - group["stoplat"].min()

    if lonspan >= latspan:
        group = group.sort_values("stoplon", ascending=(direction == 0.0))
    else:
        group = group.sort_values("stoplat", ascending=(direction == 0.0))

    rows = group[["stop_name", "stoplat", "stoplon", "obs"]].to_dict("records")

    for i in range(len(rows) - 1):
        a = rows[i]
        b = rows[i + 1]

        distm = haversine(a["stoplat"], a["stoplon"], b["stoplat"], b["stoplon"])

        if distm > 3000:
            continue

        route_edges.append({
            "route": route,
            "direction": direction,
            "from_stop": a["stop_name"],
            "to_stop": b["stop_name"],
            "from_lat": a["stoplat"],
            "from_lon": a["stoplon"],
            "to_lat": b["stoplat"],
            "to_lon": b["stoplon"],
            "distance_m": distm,
            "from_obs": a["obs"],
            "to_obs": b["obs"]
        })

route_edges_df = pd.DataFrame(route_edges)
route_edges_df.head()

,route,direction,from_stop,to_stop,from_lat,from_lon,to_lat,to_lon,distance_m,from_obs,to_obs
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,40.655490,-74.010854,40.654138,-74.008598,242.559736,1971,1854
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,40.654138,-74.008598,40.652815,-74.006473,231.866520,1854,1527
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,40.652815,-74.006473,40.651442,-74.004200,245.107279,1527,2645
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,40.651442,-74.004200,40.650020,-74.001847,253.784113,2645,553
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,40.650020,-74.001847,40.648699,-73.999661,235.769930,553,821


In [52]:
# Route-direction edge summary
route_summary = (
    route_edges_df.groupby(["route", "direction"], as_index=False)
    .agg(n_edges=("from_stop", "count"))
    .sort_values(["route", "direction"])
)

route_summary

,route,direction,n_edges
0,B35,0.0,52
1,B35,1.0,49
2,B41,0.0,50
3,B41,1.0,51
4,B6,0.0,73
5,B6,1.0,86
6,Q44-SBS,0.0,28
7,Q44-SBS,1.0,30
8,Q58,0.0,55
9,Q58,1.0,54


In [53]:
# Build total directed graph from nodes + route edges
G = nx.DiGraph()

for _, row in nodes.iterrows():
    G.add_node(
        row["stop_name"],
        stoplat=row["stoplat"],
        stoplon=row["stoplon"],
        totalobs=row["totalobs"]
    )

for _, row in route_edges_df.iterrows():
    G.add_edge(
        row["from_stop"],
        row["to_stop"],
        route=row["route"],
        direction=row["direction"],
        distance_m=row["distance_m"],
        from_obs=row["from_obs"],
        to_obs=row["to_obs"],
        weight=row["distance_m"]
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 406
Edges: 527


In [54]:
# Save outputs
nodes.to_csv("graph_nodes.csv", index=False)
route_edges_df.to_csv("graph_edges.csv", index=False)
route_summary.to_csv("graph_edge_summary.csv", index=False)

nx.write_gexf(G, "bus_stop_graph.gexf")

In [56]:
# Inspect one route if needed
route_edges_df[route_edges_df["route"] == "B35"]

,route,direction,from_stop,to_stop,from_lat,from_lon,to_lat,to_lon,distance_m,from_obs,to_obs
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,40.655490,-74.010854,40.654138,-74.008598,242.559736,1971,1854
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,40.654138,-74.008598,40.652815,-74.006473,231.866520,1854,1527
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,40.652815,-74.006473,40.651442,-74.004200,245.107279,1527,2645
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,40.651442,-74.004200,40.650020,-74.001847,253.784113,2645,553
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,40.650020,-74.001847,40.648699,-73.999661,235.769930,553,821
...,...,...,...,...,...,...,...,...,...,...,...
96,B35,1.0,39 ST/6 AV,39 ST/5 AV,40.649808,-74.001493,40.651076,-74.003594,226.488472,400,2090
97,B35,1.0,39 ST/5 AV,39 ST/4 AV,40.651076,-74.003594,40.652415,-74.005815,239.321494,2090,1620
98,B35,1.0,39 ST/4 AV,39 ST/3 AV,40.652415,-74.005815,40.653492,-74.007590,191.738362,1620,1793
99,B35,1.0,39 ST/3 AV,39 ST/2 AV,40.653492,-74.007590,40.655238,-74.010436,308.760679,1793,2223
